In [ ]:
import nibabel as nib
import numpy as np
import math
from helper import normalized_modality, format_index, get_filepath, pad_to_256
from torch.utils.data import random_split, DataLoader, Dataset
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn 
import torch
import os
from pathlib import Path
from dotenv import dotenv_values, find_dotenv

config = dotenv_values(find_dotenv(usecwd=True))
TR_DATA_PATH = Path(config.get("TR_DATA_PATH"))

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, path):
        super().__init__()
        self.path = path
        self.dirnames = sorted(os.listdir(path))
        self.dirnames = [d for d in Path(path).iterdir() if d.is_dir()]
        # UNCOMMENT (SWITCH) FOR SHORTER TRAINING (only 10 brains in dataset) - trains about 5min on GPU
        # self.len = 10
        self.len = len(self.dirnames) # 369 brains
        self.slices = 155 # slices per brain

    def __len__(self):
        return self.len * self.slices 
    
    def get_path(self, brain_index, mod):
        brain_index += 1
        return get_filepath(brain_index, mod)
    
    def get_slice(self, brain_index, slice_index, mod):
        path = self.get_path(brain_index, mod)
        img = nib.load(path)
        brain = img.get_fdata()
        if mod != "seg":
            brain = normalized_modality(brain)
        
        brain_slice = brain[:, :, slice_index]

        if mod == "seg":
            brain_slice[brain_slice == 4] = 3

        return brain_slice
    
    def __getitem__(self, index):
        brain_index = index // self.slices
        slice_index = index % self.slices

        slice = self.get_slice(brain_index, slice_index, "flair")
        seg = self.get_slice(brain_index, slice_index, "seg")

        slice = torch.tensor(slice, dtype=torch.float32).unsqueeze(0)
        slice = pad_to_256(slice)
        
        seg = torch.tensor(seg, dtype=torch.int64)  
        seg = pad_to_256(seg).squeeze(0)
        
        return slice, seg

In [3]:
train_dataset = CustomDataset(TR_DATA_PATH)

train_size = int(len(train_dataset) * 0.9)
val_size = len(train_dataset) - train_size

train_sub, val_sub = random_split(train_dataset, [train_size, val_size])
train_loader = DataLoader(train_sub, batch_size=8, shuffle=True)
val_loader = DataLoader(val_sub, batch_size=8, shuffle=False)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += y.numel()

    return total_loss / total_samples, total_correct / total_samples

In [6]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += y.numel()

    return total_loss / total_samples, total_correct / total_samples

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        # encoding
        self.econv1 = nn.Conv2d(1, 64, 3, padding=1)
        self.econv2 = nn.Conv2d(64, 64, 3, padding=1)
        
        self.econv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.econv4 = nn.Conv2d(128, 128, 3, padding=1)
        self.max2 = nn.MaxPool2d(2, 2)
        
        self.econv5 = nn.Conv2d(128, 256, 3, padding=1)
        self.econv6 = nn.Conv2d(256, 256, 3, padding=1)
        self.max3 = nn.MaxPool2d(2, 2)
        
        self.econv7 = nn.Conv2d(256, 512, 3, padding=1)
        self.econv8 = nn.Conv2d(512, 512, 3, padding=1)

        self.econv9 = nn.Conv2d(512, 1024, 3, padding=1)

        # decoding
        self.dconv1 = nn.Conv2d(1024, 1024, 3, padding=1)

        self.upconv1 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dconv2 = nn.Conv2d(1024, 512, 3, padding=1)
        self.dconv3 = nn.Conv2d(512, 512, 3, padding=1)
        
        self.upconv2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dconv4 = nn.Conv2d(512, 256, 3, padding=1)
        self.dconv5 = nn.Conv2d(256, 256, 3, padding=1)
        
        self.upconv3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dconv6 = nn.Conv2d(256, 128, 3, padding=1)
        self.dconv7 = nn.Conv2d(128, 128, 3, padding=1)

        self.upconv4 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dconv8 = nn.Conv2d(128, 64, 3, padding=1)
        self.dconv9 = nn.Conv2d(64, 64, 3, padding=1)

        self.dconv10 = nn.Conv2d(64, 4, 1)

        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.relu(self.econv1(x))
        x = self.relu(self.econv2(x))
        x1 = x.clone()
        x = self.maxpool(x)

        x = self.relu(self.econv3(x))
        x = self.relu(self.econv4(x))
        x2 = x.clone()
        x = self.maxpool(x)
        
        x = self.relu(self.econv5(x))
        x = self.relu(self.econv6(x))
        x3 = x.clone()
        x = self.maxpool(x)

        x = self.relu(self.econv7(x))
        x = self.relu(self.econv8(x))
        x4 = x.clone()
        x = self.maxpool(x)

        x = self.relu(self.econv9(x))

        x = self.relu(self.dconv1(x))

        x = self.upconv1(x)
        x = self.relu(self.dconv2(torch.cat([x4, x], dim=1)))
        x = self.relu(self.dconv3(x))

        x = self.upconv2(x)

        x = self.relu(self.dconv4(torch.cat([x3, x], dim=1)))
        x = self.relu(self.dconv5(x))

        x = self.upconv3(x)
        x = self.relu(self.dconv6(torch.cat([x2, x], dim=1)))
        x = self.relu(self.dconv7(x))

        x = self.upconv4(x)
        x = self.relu(self.dconv8(torch.cat([x1, x], dim=1)))
        x = self.relu(self.dconv9(x))

        x = self.dconv10(x)        
        return x

In [8]:
model = UNet()
model = model.to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

tr_loss_list = []
tr_acc_list = []
for epoch in range(1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    tr_loss_list.append(tr_loss)
    tr_acc_list.append(tr_acc)

print(tr_acc, tr_loss)

# FOR len(dataset) = 10, 1 epoch
# results are: acc=0.96871581812486 loss=2.9659143220613813
# high accuracy -> probably cause there is a lot of background in the scans

0.96871581812486 2.9659143220613813


In [ ]:
def aggregate_prediction(pred):
    # may be usefull for simplified segmentation
    whole_tumor = (pred > 0)
    tumor_core = np.isin(pred, [1, 3])
    enhancing = (pred == 3)
    return whole_tumor, tumor_core, enhancing
